# MUSSA: train the coronary segment model on ARCADE

Runs on a free cloud GPU (Colab: Runtime > Change runtime type > GPU; Kaggle: Accelerator > GPU).
It trains only on ARCADE, a public CC0 dataset (https://zenodo.org/records/10390295). No patient data is used or uploaded.

Steps: install, download and verify ARCADE, upload `mussa-training-kit.zip`, train, download two files and copy them to `C:\Users\<you>\MUSSA-data\models\`.

Educational research model. Not a certified medical device.

In [ ]:
!pip -q install segmentation-models-pytorch

In [ ]:
import hashlib, pathlib, urllib.request, zipfile

URL = "https://zenodo.org/records/10390295/files/arcade.zip?download=1"
EXPECTED_MD5 = "c5b1973ade06f7dff210f878161e1a76"

urllib.request.urlretrieve(URL, "/content/arcade.zip")
digest = hashlib.md5()
with open("/content/arcade.zip", "rb") as handle:
    for block in iter(lambda: handle.read(1 << 20), b""):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_MD5, f"ARCADE download is corrupt or changed: {digest.hexdigest()}"
zipfile.ZipFile("/content/arcade.zip").extractall("/content/arcade_data")
ARC = next(p.parent for p in pathlib.Path("/content/arcade_data").rglob("syntax") if p.is_dir())
print("ARCADE verified and unpacked at", ARC)

In [ ]:
from google.colab import files  # on Kaggle, add the kit as a dataset instead and set KIT below
import zipfile

uploaded = files.upload()   # choose mussa-training-kit.zip
kit_name = next(iter(uploaded))
zipfile.ZipFile(kit_name).extractall("/content/kit")
print("kit unpacked:", sorted(p.name for p in __import__("pathlib").Path("/content/kit").iterdir()))

In [ ]:
# about an hour on a free GPU; add  --smoke  first for a one-minute pipeline check
!cd /content/kit && python train_segmentation.py --arcade "{ARC}" --out /content/out

In [ ]:
import json
from google.colab import files

card = json.load(open("/content/out/segments.card.json"))
print("validation mean Dice:", card["val_mean_dice"], "| test mean Dice:", card["test_mean_dice"])
files.download("/content/out/segments.onnx")
files.download("/content/out/segments.card.json")